# Credit Score Demo

End-to-end pipeline demo using caketool:
- `feature` — auto feature generation by time window
- `model` — BoostTree (XGBoost + preprocessing pipeline)
- `experiment` — WandbTracker with dotenv_path
- `metric` — Gini, PSI
- `monitor` — Adversarial test
- `calibration` — Score calibration to normal distribution
- `report` — Risk score report

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from src.caketool.calibration import calibrate_score_to_normal
from src.caketool.experiment import create_tracker
from src.caketool.explainability import PermutationExplainer
from src.caketool.feature import generate_features_by_window
from src.caketool.metric import gini, psi
from src.caketool.model import BoostTree
from src.caketool.monitor import AdversarialModel
from src.caketool.report import decribe_risk_score

## 1. Create Synthetic Dataset

Each customer has 10–25 transactions over 180 days. Label `default=1` (~20%) is assigned based on transaction behavior + random noise.

In [ ]:
np.random.seed(42)

N_CUSTOMERS = 800
TRAIN_SIZE  = 600
REPORT_DATE = pd.Timestamp("2024-01-01")
CATEGORIES  = ["food", "transfer", "bill", "shopping", "atm"]

customer_ids = [f"C{i:04d}" for i in range(N_CUSTOMERS)]

# Assign label based on total transactions + noise
n_txns   = np.random.randint(10, 26, N_CUSTOMERS)
avg_amt  = np.random.exponential(scale=500, size=N_CUSTOMERS)
risk_score = -0.003 * avg_amt - 0.05 * n_txns + np.random.normal(0, 1, N_CUSTOMERS)
labels   = (risk_score > np.percentile(risk_score, 80)).astype(int)  # ~20% default

customer_df = pd.DataFrame({"customer_id": customer_ids, "n_txns": n_txns, "default": labels})

# Create transaction records
records = []
for _, row in customer_df.iterrows():
    scale = 200 if row["default"] == 1 else 600  # defaulters have lower amount
    for _ in range(row["n_txns"]):
        amount = np.random.exponential(scale=scale)
        records.append({
            "customer_id" : row["customer_id"],
            "report_date" : REPORT_DATE,
            "event_date"  : REPORT_DATE - pd.Timedelta(days=int(np.random.randint(1, 181))),
            "amount"      : round(amount, 2),
            "is_large_txn": amount > 1_000,
            "category"    : np.random.choice(CATEGORIES),
        })

txn_df = pd.DataFrame(records)
print(f"Transactions: {len(txn_df):,}  |  Customers: {N_CUSTOMERS}  |  Default rate: {labels.mean():.1%}")
txn_df.head(3)

## 2. Feature Engineering

Use `generate_features_by_window` to automatically aggregate features by time window (lifetime / 30d / 90d).

In [ ]:
features_df = generate_features_by_window(
    df=txn_df,
    client_id_col="customer_id",
    report_date_col="report_date",
    fs_event_timestamp="event_date",
    lookback_days=(0, 30, 90),       # 0 = lifetime, 30d, 90d
    numeric_cols=("amount",),
    boolean_cols=("is_large_txn",),
    categorical_cols=("category",),
    backend="pandas",
)

# Join label
features_df = features_df.merge(customer_df[["customer_id", "default"]], on="customer_id")

# Fill NaN = 0 for customers with no transactions in 30d/90d window
numeric_feature_cols = features_df.select_dtypes(include="number").columns
features_df[numeric_feature_cols] = features_df[numeric_feature_cols].fillna(0)

print(f"Shape: {features_df.shape}  |  Features: {features_df.shape[1] - 3}")
features_df.head(3)

## 3. Train / Test Split

In [ ]:
train_ids = set(customer_ids[:TRAIN_SIZE])
test_ids  = set(customer_ids[TRAIN_SIZE:])

non_feature_cols = {"customer_id", "report_date", "default"}
feature_cols = [
    c for c in features_df.columns
    if c not in non_feature_cols
    and not pd.api.types.is_datetime64_any_dtype(features_df[c])
]

train_df = features_df[features_df["customer_id"].isin(train_ids)]
test_df  = features_df[features_df["customer_id"].isin(test_ids)]

X_train, y_train = train_df[feature_cols], train_df["default"]
X_test,  y_test  = test_df[feature_cols],  test_df["default"]

print(f"Train: {len(X_train)} | Test: {len(X_test)} | Features: {len(feature_cols)}")
print(f"Default rate — Train: {y_train.mean():.1%} | Test: {y_test.mean():.1%}")

## 4. Train Model

`BoostTree` includes a pipeline: categorical encoding → infinity handler → univariate feature removal → collinear feature removal → XGBoost.

In [ ]:
model = BoostTree()
model.fit(X_train, y_train, eval_set=[(X_test, y_test)])

print("Top 10 features by gain:")
display(model.get_feature_importance().sort_values("gain", ascending=False).head(10))

## 5. Experiment Tracking — WandbTracker

Load credentials from `.env` via `dotenv_path`. No need to hardcode API keys.

In [ ]:
score_train = model.predict_proba(X_train)[:, 1]
score_test  = model.predict_proba(X_test)[:, 1]

gini_train = gini(y_train.values, score_train)
gini_test  = gini(y_test.values,  score_test)
psi_score  = psi(score_train, score_test)

# Only log serializable params (exclude score_func function)
model_params = {k: str(v) for k, v in model.param["model_params"].items()}

with create_tracker("wandb", "credit-score-demo", "run-01", dotenv_path="../.env") as tracker:
    tracker.log_params(model_params)
    tracker.log_metrics({
        "gini_train": round(gini_train, 4),
        "gini_test" : round(gini_test,  4),
        "psi"       : round(psi_score,  4),
    })
    print("Logged to W&B ✓")

## 6. Evaluation: Gini + PSI

- **Gini**: measures classification ability (higher = better)
- **PSI**: measures score distribution drift between train and test
  - PSI < 0.1: stable | 0.1–0.2: slight drift | > 0.2: major drift

In [ ]:
print(f"Gini train : {gini_train:.4f}")
print(f"Gini test  : {gini_test:.4f}")
print(f"PSI        : {psi_score:.4f}", end="  →  ")
if psi_score < 0.1:
    print("Stable ✓")
elif psi_score < 0.2:
    print("Slight shift ⚠")
else:
    print("Major shift ✗")

## 7. Adversarial Test

Train a classifier to distinguish train vs test. AUC ≈ 0.5 → no drift, AUC >> 0.5 → different distributions.

In [ ]:
adv = AdversarialModel()
adv.fit(X_train, X_test)
adv.show(n_features=5)

## 8. Score Calibration

`calibrate_score_to_normal` compresses extreme values toward 0.5, reducing overconfidence. Preserves rank order (monotonic).

In [ ]:
score_test_cal = calibrate_score_to_normal(score_test)

print(f"Before calibration: mean={score_test.mean():.3f}, min={score_test.min():.3f}, max={score_test.max():.3f}")
print(f"After  calibration: mean={score_test_cal.mean():.3f}, min={score_test_cal.min():.3f}, max={score_test_cal.max():.3f}")
print(f"Gini after calibration: {gini(y_test.values, score_test_cal):.4f}  (unchanged, monotonic)")

## 9. Risk Report

Analyze score by bands: default rate, approval rate, and bad/good distribution per score segment.

In [ ]:
risk_df    = pd.DataFrame({"score": score_test, "label": y_test.values})
risk_table = decribe_risk_score(risk_df, pred_col="score", label_col="label")

display(risk_table)

## 10. Model Explainability — SHAP

Use `PermutationExplainer` to understand **why** the model makes each prediction.
Works with any sklearn-compatible model — no need to extract pipeline internals.

- **Global**: which features matter most across all samples
- **Local**: how each feature contributed to a specific prediction

### 10.1 Fit the Explainer

Pass the full `BoostTree` pipeline directly — `PermutationExplainer` is model-agnostic
and uses `predict_proba` internally, so no pipeline extraction is needed.

In [ ]:
explainer = PermutationExplainer(model=model)
explainer.fit(X_test)

print(f"Samples : {X_test.shape[0]}")
print(f"Features: {X_test.shape[1]}")

### 10.2 Global Feature Importance

Mean absolute SHAP value across all test samples — a model-consistent measure of feature impact.

In [ ]:
importance_df = explainer.get_feature_importance()
display(importance_df.head(10))

### 10.3 Summary Plot

The beeswarm plot shows each sample as a dot. **Colour** encodes the feature value (red = high, blue = low). **Position on the x-axis** shows whether the feature pushed the prediction towards default (positive SHAP) or away from it (negative SHAP).

In [ ]:
explainer.show_summary(max_display=15)

### 10.4 Local Explanation

Inspect how each feature contributed to the prediction for a single customer.

In [ ]:
SAMPLE_IDX = 0

local_df = explainer.get_local_explanation(row_index=SAMPLE_IDX)

pred_prob = model.predict_proba(X_test)[SAMPLE_IDX, 1]
actual_label = y_test.iloc[SAMPLE_IDX]
print(f"Predicted probability : {pred_prob:.4f}")
print(f"Actual label          : {actual_label}")
display(local_df.head(10))

### 10.5 Waterfall Plot

Visualises how each feature moves the prediction from the base rate (expected value) to the final output for one customer.

In [ ]:
explainer.show_waterfall(row_index=SAMPLE_IDX, max_display=12)

### 10.6 Dependence Plot

Shows how the model output varies with a feature's value and highlights interaction effects.

In [ ]:
top_feature = importance_df.iloc[0]["feature"]
explainer.show_dependence(top_feature)